## MISA (2024-2025)
- Alohan'ny mamerina dia avereno atao Run ny notebook iray manontolo. Ny fanaovana azy dia redémarrena mihitsy ny kernel aloha (jereo menubar, safidio **Kernel$\rightarrow$Restart Kernel and Run All Cells**).

- Izay misy hoe `YOUR CODE HERE` na `YOUR ANSWER HERE` ihany no fenoina. Afaka manampy cells vaovao raha ilaina. Aza adino ny mameno references eo ambany raha ilaina.

## References
chatgpt,ClaudeAI


---

In [52]:
from sklearn.decomposition  import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import  load_iris, load_breast_cancer, make_blobs
import numpy as np
from random import randrange
from sklearn.metrics import accuracy_score
import cvxpy as cp

def grad_check_sparse(f, x, analytic_grad, num_checks=12, h=1e-5, error=1e-9):
    """
    sample a few random elements and only return numerical
    in this dimensions
    """

    for i in range(num_checks):
        ix = tuple([randrange(m) for m in x.shape])

        oldval = x[ix]
        x[ix] = oldval + h  # increment by h
        fxph = f(x)  # evaluate f(x + h)
        x[ix] = oldval - h  # increment by h
        fxmh = f(x)  # evaluate f(x - h)
        x[ix] = oldval  # reset

        grad_numerical = (fxph - fxmh) / (2 * h)
        grad_analytic = analytic_grad[ix]
        rel_error = abs(grad_numerical - grad_analytic) / (
            abs(grad_numerical) + abs(grad_analytic)
        )
        print(
            "numerical: %f analytic: %f, relative error: %e"
            % (grad_numerical, grad_analytic, rel_error)
        )
        assert rel_error < error

def rel_error(x, y):
    """ returns relative error """
    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))

data = load_iris()
X, y = data.data, data.target

# PCA

In [53]:
class PrincipalComponentAnalysis():
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
    
    def fit(self, X):
        """
        Fit PCA to the data X using Singular Value Decomposition (SVD).
        
        Args:
            X: Input data of shape (N, D) that should be centered
        
        Returns:
            self
        """

        U, S, Vt = np.linalg.svd(X, full_matrices=False)
        
        V = Vt.T  
        
        # Store only the first n_components principal components
        self.components = V[:, :self.n_components] 
        
        return self
    
    def transform(self, X):
        if self.components is None:
            raise ValueError("PCA model has not been fitted yet.")
        # Project data onto the principal components
        X_transformed = X @ self.components 
        
        return X_transformed

In [54]:
X_centered = X - np.mean(X, axis=0)

pca = PCA(n_components=3)
pca.fit(X_centered)
X_pca_trans = pca.transform(X_centered)

model = PrincipalComponentAnalysis(n_components=3)
model.fit(X_centered)
X_model_trans = model.transform(X_centered)

sign_correct_X_model_trans = np.concatenate([np.expand_dims(X_model_trans[:,0],axis=1),-X_model_trans[:,1:]],axis=1)

error = rel_error(X_pca_trans, sign_correct_X_model_trans)
print(error)
assert  error < 1e-11

1.606305433281053e-13


# Binary Linear SVM with CVXPY

## Hard margin 

In [55]:
X2, y2 = make_blobs(n_samples=300, centers=2, n_features=12, random_state=47)
scaler = StandardScaler()
X2 = scaler.fit_transform(X2)
y2[y2 == 0] = -1

$$\min_{\mathbf{w},b}\frac{1}{2}||\mathbf{w}||^2$$ <br>
$$\text{s.t } y_i(\mathbf{w}^{\top}\mathbf{x}_i + b) \ge 1, \ i=1...N$$

In [56]:
class LinearSVMHardMargin():
    def __init__(self):
        self.w = None
        self.b = 0
    
    def fit(self, X, y):

        N, D = X.shape
        
        w = cp.Variable(D)
        b = cp.Variable()
        
        objective = cp.Minimize(0.5 * cp.sum_squares(w))
        
        constraints = [cp.multiply(y, X @ w + b) >= 1]

        problem = cp.Problem(objective, constraints)
        problem.solve(verbose=False)
        
        self.w = w.value
        self.b = b.value
        
        return self
        
    def predict(self, X):
        if self.w is None:
            raise ValueError("Model has not been fitted yet.")
        
        decision_values = X @ self.w + self.b
        predictions = np.sign(decision_values)
        
        return predictions

In [57]:
model = LinearSVMHardMargin()
model.fit(X2, y2)
pred = model.predict(X2)
accuracy = accuracy_score(y2, pred)
print(accuracy)
assert accuracy == 1

1.0


## Soft Margin

In [58]:
data3 = load_breast_cancer()
X3, y3 = data3.data, data3.target
scaler = StandardScaler()
X3 = scaler.fit_transform(X3)
y3[y3 == 0] = -1

$$L(\mathbf{w},b) = \frac{1}{N} \sum_{i=1}^N \max(0, 1 - y_i(\mathbf{w}^{\top}\mathbf{x}_i + b)) + \lambda||\mathbf{w}||^2_2$$

In [59]:
class LinearSVMSoftMargin():
    def __init__(self, alpha=0):
        self.w = None
        self.b = 0
        self.alpha = alpha
    
    def fit(self, X, y):
        N, D = X.shape
        
        w = cp.Variable(D)
        b = cp.Variable()
        
        hinge_loss = cp.sum(cp.pos(1 - cp.multiply(y, X @ w + b)))
        reg_term = self.alpha * cp.norm(w, 2)**2
        
        objective = cp.Minimize((1/N) * hinge_loss + reg_term)
        
        prob = cp.Problem(objective)
        prob.solve()
        
        self.w = w.value
        self.b = b.value
        
    def predict(self, X):
        scores = X @ self.w + self.b
        y_pred = np.sign(scores)
        
        y_pred[y_pred == 0] = -1
        return y_pred

In [60]:
model = LinearSVMSoftMargin(alpha=1e-3)
model.fit(X3, y3)
pred = model.predict(X3)
accuracy = accuracy_score(y3, pred)
print(accuracy)
assert accuracy >= 0.98

0.9876977152899824


# Multiclass Linear SVM

## Loss

$$L(\mathbf{W}) = \sum_{i=1}^N \sum_{j \neq y_i} \max(0, s_j - s_{y_i} + 1) + \lambda||\mathbf{w}||^2_2$$ <br>
$$\text{where } s_j = (f(\mathbf{x}_i;\mathbf{W}))_j = (\mathbf{W}\mathbf{x}_i)_j \text{ is the score for the j-th class}$$

In [61]:
W = np.random.randn(X.shape[1], 3) * 0.0001

In [62]:
def svm_loss_naive(W, X, y, alpha):
    """
    Multiclass SVM loss function WITH FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    
    # Initialization
    loss = 0.0
    dW = np.zeros_like(W)
    N, D = X.shape
    C = W.shape[1] 
    
    scores = X @ W
    
    # Loop over all examples
    for i in range(N):
        correct_class = y[i]
        s_correct = scores[i, correct_class]
        
        # Loop over all classes
        for j in range(C):
            if j != correct_class:
                margin_loss = scores[i, j] - s_correct + 1
                
                if margin_loss > 0:
                    loss += margin_loss
                    
                    dW[:, j] += X[i]  
                    dW[:, correct_class] -= X[i] 
    
    loss /= N
    dW /= N
    
    loss += alpha * np.sum(W * W)
    dW += 2 * alpha * W

    return loss, dW

In [63]:
# NO REGLARIZATION
loss, dW = svm_loss_naive(W, X, y, 0.0)

f = lambda W: svm_loss_naive(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -0.370667 analytic: -0.370667, relative error: 6.629334e-11
numerical: -0.744667 analytic: -0.744667, relative error: 3.954066e-11
numerical: 0.953333 analytic: 0.953333, relative error: 1.861856e-11
numerical: -0.370667 analytic: -0.370667, relative error: 6.629334e-11
numerical: -0.126667 analytic: -0.126667, relative error: 8.787894e-11
numerical: 0.837333 analytic: 0.837333, relative error: 2.002180e-12
numerical: 0.953333 analytic: 0.953333, relative error: 1.861856e-11
numerical: 2.296000 analytic: 2.296000, relative error: 3.549232e-13
numerical: -0.092667 analytic: -0.092667, relative error: 4.556689e-10
numerical: 0.287333 analytic: 0.287333, relative error: 1.232111e-10
numerical: -0.744667 analytic: -0.744667, relative error: 3.954066e-11
numerical: 0.287333 analytic: 0.287333, relative error: 1.232111e-10


In [64]:
#REGLARIZATION
loss, dW = svm_loss_naive(W, X, y, 2)

f = lambda W: svm_loss_naive(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: 0.953638 analytic: 0.953638, relative error: 1.678802e-11
numerical: -0.093061 analytic: -0.093061, relative error: 4.494003e-10
numerical: -0.369684 analytic: -0.369684, relative error: 6.789866e-11
numerical: -0.826266 analytic: -0.826266, relative error: 1.655594e-12
numerical: 0.953638 analytic: 0.953638, relative error: 1.678802e-11
numerical: 0.837035 analytic: 0.837035, relative error: 3.478288e-12
numerical: 0.288501 analytic: 0.288501, relative error: 1.189213e-10
numerical: 2.295716 analytic: 2.295716, relative error: 1.187640e-12
numerical: -0.126778 analytic: -0.126778, relative error: 7.781672e-11
numerical: -0.826266 analytic: -0.826266, relative error: 1.655594e-12
numerical: -1.794337 analytic: -1.794337, relative error: 6.611579e-12
numerical: -0.500690 analytic: -0.500690, relative error: 2.000935e-11


In [65]:
def svm_loss_vectorized(W, X, y, alpha):
    """
    Multiclass SVM loss function WITHOUT FOR LOOPS

    Inputs:
    - W: array of shape (D, C) containing weights
    - X: array of shape (N, D) containing a minibatch of data
    - y: array of shape (N,) containing training labels
    - alpha: (float) regularization 

    Returns a tuple of:
    - loss as single float
    - gradient with respect to weights W;  same shape as W
    """
    loss = 0.0
    dW = np.zeros_like(W)
    
    N, D = X.shape
    C = W.shape[1]  
    
    scores = X @ W  
    correct_class_scores = scores[np.arange(N), y]  
    
    margins = scores - correct_class_scores[:, np.newaxis] + 1 
    margins[np.arange(N), y] = 0
    
    hinge_losses = np.maximum(0, margins) 
    loss = np.sum(hinge_losses) / N
    
    loss_mask = (margins > 0).astype(float)  
    loss_count = np.sum(loss_mask, axis=1)  
    loss_mask[np.arange(N), y] = -loss_count  
    
    dW = (X.T @ loss_mask) / N
    
    loss += alpha * np.sum(W * W)
    dW += 2 * alpha * W

    return loss, dW

In [66]:
# NO REGLARIZATION
loss, dW = svm_loss_vectorized(W, X, y, 0.0)

f = lambda W: svm_loss_vectorized(W, X, y, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: 0.287333 analytic: 0.287333, relative error: 5.066491e-11
numerical: 0.953333 analytic: 0.953333, relative error: 1.149896e-12
numerical: -0.126667 analytic: -0.126667, relative error: 2.312843e-13
numerical: 0.953333 analytic: 0.953333, relative error: 1.149896e-12
numerical: 2.296000 analytic: 2.296000, relative error: 3.549232e-13
numerical: 2.296000 analytic: 2.296000, relative error: 3.549232e-13
numerical: -0.126667 analytic: -0.126667, relative error: 2.312843e-13
numerical: 0.287333 analytic: 0.287333, relative error: 5.066491e-11
numerical: -0.370667 analytic: -0.370667, relative error: 8.587256e-12
numerical: -0.744667 analytic: -0.744667, relative error: 2.269820e-12
numerical: 0.083333 analytic: 0.083333, relative error: 3.273715e-12
numerical: 2.296000 analytic: 2.296000, relative error: 3.549232e-13


In [67]:
# REGLARIZATION
loss, dW = svm_loss_vectorized(W, X, y, 2)

f = lambda W: svm_loss_vectorized(W, X, y, 2)[0]
grad_numerical = grad_check_sparse(f, W, dW, error=1e-9)

numerical: -0.126778 analytic: -0.126778, relative error: 9.753895e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.826266 analytic: -0.826266, relative error: 1.655796e-12
numerical: 0.288501 analytic: 0.288501, relative error: 5.425096e-11
numerical: -0.500690 analytic: -0.500690, relative error: 2.165609e-12
numerical: 0.083254 analytic: 0.083254, relative error: 1.074252e-11
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12
numerical: -0.369684 analytic: -0.369684, relative error: 7.180946e-12


## Gradient descent

In [68]:
class LinearModel():
    def __init__(self, fit_intercept=True):
        self.W = None
        self.fit_intercept = fit_intercept

    def train(self, X, y, learning_rate=1e-3, alpha=0, num_iters=100, batch_size=200, verbose=False):
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([X, ones])
            
        N, d = X.shape
        C = (np.max(y) + 1) 
        
        if self.W is None:
            self.W = 0.001 * np.random.randn(d, C)

        loss_history = []
        for it in range(num_iters):
            indices = np.random.choice(N, batch_size, replace=True)
            X_batch = X[indices]
            y_batch = y[indices]
            
            loss, dW = self.loss(X_batch, y_batch, alpha)
            loss_history.append(loss)

            self.W -= learning_rate * dW
            
            if verbose and it % 10000 == 0:
                print("iteration %d / %d: loss %f" % (it, num_iters, loss))
                
        return loss_history

    def predict(self, X):
        pass

    def loss(self, X_batch, y_batch, reg):
        pass

class LinearSVM(LinearModel):
    """ Multiclass SVM """

    def loss(self, X_batch, y_batch, alpha):
        return svm_loss_vectorized(self.W, X_batch, y_batch, alpha)
    
    def predict(self, X):
        """ 
        Returns:
        - y_pred: predicted class indices
        """
        if self.fit_intercept:
            ones = np.ones((X.shape[0], 1))
            X = np.hstack([X, ones])
        
        scores = X.dot(self.W) 
        
        y_pred = np.argmax(scores, axis=1)
        
        return y_pred

In [69]:
model = LinearSVM(fit_intercept=True)
model.train(X, y, num_iters=75000, batch_size=64, learning_rate=1e-3, verbose=True)
pred = model.predict(X)
model_accuracy = accuracy_score(y, pred)
print(model_accuracy)
assert model_accuracy > 0.97

iteration 0 / 75000: loss 2.001441


iteration 10000 / 75000: loss 0.095392
iteration 20000 / 75000: loss 0.106465
iteration 30000 / 75000: loss 0.049013
iteration 40000 / 75000: loss 0.042507
iteration 50000 / 75000: loss 0.028350
iteration 60000 / 75000: loss 0.063069
iteration 70000 / 75000: loss 0.152395
0.9733333333333334
